In [1]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test') \
        .config('spark.driver.memory', '8g') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 11:49:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/05 11:49:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/05 11:49:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
df_green = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/green/')

In [4]:
df_green.createOrReplaceTempView("green")

In [10]:
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    -- Revenue calculation 
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [11]:
df_green_revenue.show()

[Stage 8:==============>                                          (4 + 12) / 16]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|   7|            769.73|            45|
|2020-01-01 00:00:00|  17|195.03000000000003|             9|
|2020-01-01 00:00:00|  18|               7.8|             1|
|2020-01-01 00:00:00|  22|              15.8|             1|
|2020-01-01 00:00:00|  24|              87.6|             3|
|2020-01-01 00:00:00|  25|             531.0|            26|
|2020-01-01 00:00:00|  29|              61.3|             1|
|2020-01-01 00:00:00|  32| 68.94999999999999|             2|
|2020-01-01 00:00:00|  33|            317.27|            11|
|2020-01-01 00:00:00|  35|            129.96|             5|
|2020-01-01 00:00:00|  36|295.34000000000003|            11|
|2020-01-01 00:00:00|  37|            175.67|             6|
|2020-01-01 00:00:00|  38| 98.78999999999999|             2|
|2020-01-01 00:00:00|  4

In [12]:
df_green_revenue.write.parquet('data/report/revenue/green/')

In [14]:
df_yellow = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/yellow/')

In [15]:
df_yellow.createOrReplaceTempView("yellow")

In [16]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    -- Revenue calculation 
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [22]:
df_yellow_revenue.repartition(20).write.parquet('data/report/revenue/yellow/', mode="overwrite")

In [24]:
df_green_revenue.repartition(20).write.parquet('data/report/revenue/green/', mode="overwrite")